# Phase 6 — SQL Analytics

This notebook queries the CMS Medicare SQLite database directly using SQL.
While Phase 5 used Python and Pandas to visualize billing patterns,
Phase 6 goes deeper — using SQL to answer specific business questions
that surface in real healthcare fraud investigations.

## Why SQL after Python?
SQL is the language of databases and business intelligence. Every
healthcare analytics company runs SQL queries against their data
warehouse daily. This phase demonstrates that capability on real
government data.

## 6 Business Questions
1. Which providers are billing for procedures outside their specialty scope?
2. Which state and specialty combinations drive the highest billing?
3. Which providers have the largest gap between charge and payment?
4. Can we detect billing anomalies using pure SQL?
5. Which providers see the most patients vs generate the most revenue?
6. How does procedure cost vary across specialties?

In [1]:
import sqlite3
import pandas as pd

# Connect to our SQLite database
conn = sqlite3.connect('data/cms_medicare.db')

# Test connection
test = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM medicare_billing", conn)
print(f"Database connected successfully")
print(f"Total rows in database: {test['total_rows'][0]:,}")

Database connected successfully
Total rows in database: 500,000


## Query 1 - Scope of Practice Violations

Identifying providers billing for procedures outside their licensed 
scope of practice. This is one of the most common forms of Medicare 
fraud - billing for services a provider is not qualified to perform.


We focus on non-surgical specialties billing for high-cost surgical 
procedure codes, which represent the clearest scope violations.

In [2]:
# Query 1 - Scope of Practice Violations
query1 = """
SELECT 
    provider_name,
    first_name,
    specialty,
    procedure_code,
    procedure_desc,
    avg_submitted_charge,
    avg_medicare_payment,
    total_patients,
    state
FROM medicare_billing
WHERE 
    specialty IN (
        'Nurse Practitioner',
        'Physician Assistant',
        'Family Practice',
        'Internal Medicine',
        'Psychiatry',
        'Dermatology'
    )
    AND (
        procedure_desc LIKE '%surgery%'
        OR procedure_desc LIKE '%fusion%'
        OR procedure_desc LIKE '%replacement%'
        OR procedure_desc LIKE '%reconstruction%'
        OR procedure_desc LIKE '%transplant%'
        OR procedure_desc LIKE '%amputation%'
    )
    AND avg_submitted_charge > 1000
ORDER BY avg_submitted_charge DESC
LIMIT 20
"""

scope_violations = pd.read_sql_query(query1, conn)
print(f"Potential scope violations found: {len(scope_violations)}")
print(scope_violations[['provider_name', 'specialty',
                         'procedure_desc',
                         'avg_submitted_charge',
                         'state']].to_string(index=False))

Potential scope violations found: 20
provider_name           specialty                                                                              procedure_desc  avg_submitted_charge state
  Ellenberger  Nurse Practitioner                   Fusion of spine in lower back with partial removal of spine bone and disc          28056.083333    NJ
      Francis Physician Assistant                                               Replacement of knee joint, both sides of knee          22819.650000    NJ
       Kawash Physician Assistant                                               Replacement of knee joint, both sides of knee          22187.333333    NJ
  Chedalavada Physician Assistant                                     Replacement of thigh bone and hip joint with prosthesis          21500.000000    MD
   Stuedemann  Nurse Practitioner                                     Replacement of thigh bone and hip joint with prosthesis          17652.000000    IL
       Lehrer Physician Assistant      

## Query 1 - Key Findings: Scope of Practice Violations

Query 1 returned 20 potential violations across the dataset. Looking 
through the results, the violations are heavily concentrated among 
Nurse Practitioners and Physician Assistants billing for major surgical 
procedures like spinal fusion and bilateral knee replacement — procedures 
that fall strictly outside their licensed scope of practice.

What stands out geographically is that NJ, MD, and WI appear repeatedly 
in the results. This connects directly back to the geographic analysis 
in Phase 5 where DC, MD, and NJ ranked as the highest billing states. 
At the time we attributed this to cost of living and specialist 
concentration. This query adds another dimension — scope violations 
are disproportionately concentrated in the same high billing states, 
which may be partly driving those elevated averages.

Ellenberger, a Nurse Practitioner from NJ billing $28,056 for spinal 
fusion surgery, appears as the top offender. This is the same provider 
flagged by the z-score anomaly detection in Phase 5. Two completely 
independent methods — statistical outlier detection in Python and scope 
keyword matching in SQL — both flagging the same provider significantly 
strengthens the case for investigation.

## Query 2 — State and Specialty Billing Breakdown

Phase 5 identified DC and NJ as the highest billing states overall.
This query goes deeper — finding which specific specialty within 
each state is driving those elevated averages.

This is the difference between a surface level observation and 
an actionable insight. Knowing a state bills high is interesting. 
Knowing exactly which specialty in which state is responsible 
gives investigators a specific target.

In [3]:
# Query 2 - State and Specialty Breakdown
query2 = """
SELECT 
    state,
    specialty,
    COUNT(DISTINCT provider_id) as total_providers,
    ROUND(AVG(avg_submitted_charge), 2) as avg_charge,
    ROUND(AVG(avg_medicare_payment), 2) as avg_payment,
    ROUND(SUM(avg_medicare_payment), 2) as total_payment,
    ROUND(AVG(avg_submitted_charge) - AVG(avg_medicare_payment), 2) as avg_gap
FROM medicare_billing
WHERE state IN ('DC', 'NJ', 'AK', 'NY', 'CT')
GROUP BY state, specialty
HAVING COUNT(DISTINCT provider_id) >= 5
ORDER BY state, avg_charge DESC
LIMIT 30
"""

state_specialty = pd.read_sql_query(query2, conn)
print(f"Results found: {len(state_specialty)}")
print(state_specialty.to_string(index=False))

Results found: 30
state                                     specialty  total_providers  avg_charge  avg_payment  total_payment  avg_gap
   AK Certified Registered Nurse Anesthetist (CRNA)                8     1156.29       141.54        3679.98  1014.75
   AK                            Emergency Medicine                6      755.77       123.05        2707.20   632.72
   AK                           Physician Assistant               17      409.28        63.45        6725.28   345.84
   AK                            Nurse Practitioner               21      318.52        64.84        5511.43   253.68
   AK                             Internal Medicine                5      284.31        70.07        2732.91   214.24
   AK                                     Optometry                7      241.97        65.62        3346.66   176.35
   AK                               Family Practice               11      221.64        63.14        6313.56   158.51
   AK        Physical Therapist in Pri

## Query 2 — Key Findings: State and Specialty Breakdown

Query 2 returned 30 state and specialty combinations across DC, NJ, 
AK, NY and CT. Looking at DC specifically, Anesthesiology stands out 
with only 5 providers but a higher average charge than more complex 
specialties like Plastic and Reconstructive Surgery.

What makes this interesting is the volume vs value dynamic. Radiology 
generates high total payments not because each procedure is expensive 
but because one patient visit can trigger multiple separate billing 
codes — every scan, every body part billed separately. Anesthesiology 
works the opposite way — fewer patients but each procedure is expensive 
because billing is time based.

The concern with DC Anesthesiology is that just 5 providers are 
averaging $1,373 per procedure — slightly above the national average 
we identified in Phase 5. A small group of providers all billing at 
the top of their specialty range in a concentrated geographic area 
is worth flagging for closer review.

## Query 3 - Highest Charge vs Payment Gap by Specialty

Identifying which specialties have the largest difference between 
what providers submit and what Medicare actually pays. A gap 
dramatically above the specialty norm warrants investigation.

In [4]:
# Query 3 - Charge vs Payment Gap Analysis
query3 = """
SELECT 
    specialty,
    COUNT(DISTINCT provider_id) as total_providers,
    ROUND(AVG(avg_submitted_charge), 2) as avg_charge,
    ROUND(AVG(avg_medicare_payment), 2) as avg_payment,
    ROUND(AVG(avg_submitted_charge) - AVG(avg_medicare_payment), 2) as avg_gap,
    ROUND((AVG(avg_submitted_charge) - AVG(avg_medicare_payment)) 
          / AVG(avg_submitted_charge) * 100, 1) as gap_percentage,
    ROUND(MAX(avg_submitted_charge), 2) as max_charge,
    ROUND(MIN(avg_submitted_charge), 2) as min_charge
FROM medicare_billing
GROUP BY specialty
HAVING COUNT(DISTINCT provider_id) >= 10
ORDER BY gap_percentage DESC
LIMIT 15
"""

gap_analysis = pd.read_sql_query(query3, conn)
print(f"Specialties analyzed: {len(gap_analysis)}")
print(gap_analysis[['specialty', 'total_providers', 
                     'avg_charge', 'avg_payment',
                     'gap_percentage', 
                     'max_charge']].to_string(index=False))

Specialties analyzed: 15
                                     specialty  total_providers  avg_charge  avg_payment  gap_percentage  max_charge
                      Anesthesiology Assistant              116     1123.11        78.71            93.0     4594.57
                                Anesthesiology             1739     1316.32       100.88            92.3    34800.00
 Certified Registered Nurse Anesthetist (CRNA)             2039     1255.47       103.32            91.8    19392.55
                Interventional Pain Management               63      768.21        97.85            87.3    30000.00
                            Emergency Medicine             2388      679.98        89.67            86.8     7299.00
Independent Diagnostic Testing Facility (IDTF)              103      853.79       118.92            86.1     9600.00
                      Interventional Radiology              113      803.57       113.82            85.8    58405.00
                          Diagnostic Ra

## Query 3 - Key Findings: Charge vs Payment Gap

The top 3 results are all anesthesia related specialties clustering 
between 91 and 93% gap percentage. This consistency across all three 
anesthesia types confirms this is an industry norm rather than 
suspicious behavior — they all follow the same Medicare payment 
rules for anesthesia billing.

However gap percentage alone is not enough to confirm or dismiss 
fraud. It is a useful starting point for flagging anomalies but 
the next step is always to examine what specific procedures are 
driving that gap. A single anesthesiologist at 97% when peers 
average 92% is where the real red flag appears.

## Query 4 - Anomaly Detection in Pure SQL

Replicating the z-score anomaly detection from Phase 5 using pure SQL.
This demonstrates that the same analysis can be done directly in the 
database without loading data into Python first — which is how it 
would run in a production healthcare analytics system.

In [5]:
# Query 4 - Anomaly Detection in Pure SQL
query4 = """
WITH specialty_stats AS (
    SELECT 
        specialty,
        AVG(avg_submitted_charge) as specialty_mean,
        AVG(avg_submitted_charge * avg_submitted_charge) - 
            AVG(avg_submitted_charge) * AVG(avg_submitted_charge) as variance
    FROM medicare_billing
    GROUP BY specialty
),
provider_stats AS (
    SELECT
        m.provider_name,
        m.first_name,
        m.specialty,
        m.state,
        AVG(m.avg_submitted_charge) as provider_avg_charge,
        s.specialty_mean,
        SQRT(s.variance) as specialty_std,
        ROUND((AVG(m.avg_submitted_charge) - s.specialty_mean) / 
              NULLIF(SQRT(s.variance), 0), 2) as z_score
    FROM medicare_billing m
    JOIN specialty_stats s ON m.specialty = s.specialty
    GROUP BY m.provider_name, m.first_name, m.specialty, m.state
)
SELECT *
FROM provider_stats
WHERE z_score > 2
ORDER BY z_score DESC
LIMIT 15
"""

anomalies = pd.read_sql_query(query4, conn)
print(f"Anomalies detected: {len(anomalies)}")
print(anomalies[['provider_name', 'specialty', 'state',
                  'provider_avg_charge', 'specialty_mean',
                  'z_score']].to_string(index=False))

Anomalies detected: 15
        provider_name                    specialty state  provider_avg_charge  specialty_mean  z_score
              Johnson            Internal Medicine    TX         23866.666667      226.184981    69.16
               Taylor                 Chiropractic    AZ          1190.000000       58.855207    34.92
     Personalis, Inc.          Clinical Laboratory    CA          8455.000000      103.265020    30.52
               Kawash          Physician Assistant    NJ         22187.333333      338.536742    28.31
          Ellenberger           Nurse Practitioner    NJ         10140.465278      219.989503    25.64
       Healthy Rx Inc Mass Immunizer Roster Biller    NY          1664.787692       81.050262    19.05
Biotheranostics, Inc.          Clinical Laboratory    CA          4432.126267      103.265020    15.82
                Ellis           Nurse Practitioner    FL          6081.818182      219.989503    15.15
                Jones          Physician Assistant

In [6]:
# Investigate Johnson
johnson_query = """
SELECT 
    procedure_desc,
    avg_submitted_charge,
    total_patients
FROM medicare_billing
WHERE provider_name = 'Johnson'
AND specialty = 'Internal Medicine'
ORDER BY avg_submitted_charge DESC
"""

johnson = pd.read_sql_query(johnson_query, conn)
print(johnson.to_string(index=False))

                                                                                                                      procedure_desc  avg_submitted_charge  total_patients
                                                           Emergency department visit with moderate level of medical decision making          23866.666667              12
                                                                                                  Critical care, first 30-74 minutes           2690.000000              24
                                                               Emergency department visit with high level of medical decision making           2089.000000             135
                                                           Emergency department visit with moderate level of medical decision making           1416.000000              54
                                                                                                  Critical care, first 30-74 minutes            7

## Query 4 - Key Findings: Anomaly Detection in Pure SQL

**Johnson - Internal Medicine, TX (z-score: 69)**
Johnson's average charge of USD 23,866 sits against a specialty average 
of USD 226 — a 105x deviation. While Internal Medicine physicians do treat 
complex chronic conditions, case complexity alone does not justify this 
gap. Drilling into Johnson's specific procedures revealed the source: 
a standard emergency department visit billed at USD 23,866 when the same 
code typically costs USD 200-500. Every other procedure Johnson bills is 
priced normally. This is a textbook example of upcoding — billing a 
legitimate procedure at a fraudulently inflated rate.

**Taylor - Chiropractic, AZ (z-score: 34)**
Taylor's average charge of USD 1,190 against a specialty average of USD 58 
is unusual for a field built around low cost joint adjustments and 
spinal manipulation. This suggests Taylor may be billing procedure 
codes outside the normal scope of chiropractic services.

**Personalis Inc - Clinical Laboratory, CA (z-score: 30)**
As a laboratory entity rather than an individual provider, Personalis 
billing USD 8,455 on average could reflect legitimate advanced genomic 
sequencing tests which can cost thousands. However the deviation from 
typical lab billing warrants closer examination to rule out overbilling.

The key takeaway across all three cases — z-score flags the anomaly, 
but procedure level investigation determines whether it is fraud. 
Johnson's drill-down is demonstrated below.

## Query 5 — Volume vs Value: Top Providers by Patients and Revenue

Not all high billing providers are suspicious. Some see massive patient 
volumes and bill legitimately. This query separates two types of providers:

Volume players — see the most patients, lower revenue per patient
Value players — see fewer patients but generate high revenue per patient

A provider with high revenue but very few patients is a red flag.
A provider with high volume and proportional revenue is legitimate.

In [8]:
# Query 5 - Volume vs Value Analysis
query5 = """
SELECT 
    provider_name,
    first_name,
    specialty,
    state,
    COUNT(DISTINCT procedure_desc) as unique_procedures,
    SUM(total_patients) as total_patients,
    ROUND(SUM(avg_medicare_payment), 2) as total_revenue,
    ROUND(SUM(avg_medicare_payment) / NULLIF(SUM(total_patients), 0), 2) as revenue_per_patient
FROM medicare_billing
GROUP BY provider_name, first_name, specialty, state
HAVING SUM(total_patients) >= 100
ORDER BY revenue_per_patient DESC
LIMIT 15
"""

volume_value = pd.read_sql_query(query5, conn)
print(f"Results: {len(volume_value)}")
print(volume_value[['provider_name', 'specialty', 'state',
                     'total_patients', 'total_revenue',
                     'revenue_per_patient']].to_string(index=False))

Results: 15
                                 provider_name                  specialty state  total_patients  total_revenue  revenue_per_patient
               Phoenix Eye Surgical Center Plc Ambulatory Surgical Center    AZ             112       41183.14               367.71
   Advanced Spine And Pain Surgery Centers Llc Ambulatory Surgical Center    AZ             187       41641.32               222.68
                                          Nedd           Thoracic Surgery    DC             152       30669.81               201.78
        Lakeway Ambulatory Surgery Center, Llc Ambulatory Surgical Center    TX             167       27416.62               164.17
                New England Surgery Center Llc Ambulatory Surgical Center    MA             509       73943.24               145.27
            Overland Park Surgical Suites, Llc Ambulatory Surgical Center    KS             122       16959.00               139.01
                  Carlsbad Surgery Center, Llc Ambulatory Surgic

In [9]:
# Investigate Phoenix Eye Surgical Center
phoenix_query = """
SELECT 
    procedure_desc,
    avg_submitted_charge,
    avg_medicare_payment,
    total_patients
FROM medicare_billing
WHERE provider_name LIKE '%Phoenix Eye%'
ORDER BY avg_submitted_charge DESC
"""

phoenix = pd.read_sql_query(phoenix_query, conn)
print(phoenix.to_string(index=False))

                                                  procedure_desc  avg_submitted_charge  avg_medicare_payment  total_patients
       Insertion of spinal neurostimulator generator or receiver               72500.0          18669.182286              34
    Insertion of peripheral or gastric neurostimulator generator               55300.0          14973.110000              12
       Insertion of sacral nerve neurostimulator electrode array               14021.0           3863.173816              36
Insertion of spinal neurostimulator electrode array through skin               13715.0           3677.669000              30


## Query 5 - Key Findings: Volume vs Value Analysis

Looking at the results, multiple Ambulatory Surgical Centers dominate 
the top revenue per patient rankings. Comparing the first and last row 
revealed an interesting contrast — Phoenix Eye Surgical Center sees 
only 112 patients but generates USD 367 per patient, while Advanced 
Surgery Center of Lancaster sees 947 patients at only USD 95 per patient.

This difference made me curious so I drilled into Phoenix Eye's actual 
procedures. Despite being named an eye surgical center, their highest 
billed procedures are spinal neurostimulator insertions — completely 
unrelated to eye care. A facility named Phoenix Eye Surgical Center 
has no legitimate reason to be performing spinal surgery.

This points to one of two scenarios — either the center changed 
specialties but never updated their Medicare billing profile, or 
they are billing for procedures not actually performed at that location. 
Either way this is a facility level anomaly that warrants investigation, 
separate from the individual provider fraud patterns identified earlier.

## Query 6 - Procedure Cost Benchmarking

This query establishes cost benchmarks for the most common procedures 
across all specialties. Benchmarking answers a simple but powerful question:

What should a procedure cost?

Once you know the benchmark, any provider billing significantly above 
it becomes an immediate target for investigation. This is the foundation 
of the Text-to-SQL AI layer coming in Phase 7 — when someone asks 
"is this charge normal?" the AI compares against these benchmarks.

In [11]:
# Query 6 - Procedure Cost Benchmarking
query6 = """
SELECT 
    procedure_desc,
    COUNT(DISTINCT provider_id) as total_providers,
    ROUND(AVG(avg_submitted_charge), 2) as benchmark_charge,
    ROUND(AVG(avg_medicare_payment), 2) as benchmark_payment,
    ROUND(MIN(avg_submitted_charge), 2) as min_charge,
    ROUND(MAX(avg_submitted_charge), 2) as max_charge,
    ROUND(MAX(avg_submitted_charge) - MIN(avg_submitted_charge), 2) as charge_range,
    SUM(total_patients) as total_patients_nationwide
FROM medicare_billing
GROUP BY procedure_desc
HAVING COUNT(DISTINCT provider_id) >= 50
ORDER BY total_patients_nationwide DESC
LIMIT 15
"""

benchmarks = pd.read_sql_query(query6, conn)
print(f"Procedures benchmarked: {len(benchmarks)}")
print(benchmarks[['procedure_desc', 'total_providers',
                   'benchmark_charge', 'benchmark_payment',
                   'min_charge', 'max_charge',
                   'charge_range']].to_string(index=False))

Procedures benchmarked: 15
                                                                                                                           procedure_desc  total_providers  benchmark_charge  benchmark_payment  min_charge  max_charge  charge_range
                                                                      Established patient office or other outpatient visit, 30-39 minutes            24662            266.64              80.02       62.12     1945.50       1883.38
                                                                      Established patient office or other outpatient visit, 20-29 minutes            23564            184.60              56.11       42.38     1426.00       1383.62
                                                                             Insertion of needle into vein for collection of blood sample             4465             16.44               8.01        2.72      135.62        132.90
                               Subsequent hospital ca